In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from langchain_community.graphs import Neo4jGraph

In [ ]:
import sys
import os

# Get the current directory of the notebook (exploration/)
current_dir = os.path.dirname(os.path.abspath("explore_graph.ipynb"))

# Get the parent directory (my_project/)
parent_dir = os.path.dirname(current_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

In [ ]:
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,NEO4J_DATABASE, DIRECTORY
graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE)

In [ ]:
# prod
graph.query(
    "MATCH (n:__Community__)"
    "RETURN count(*)"
)

In [ ]:
# dev
graph.query(
    "MATCH (n:__Community__)"
    "RETURN count(*)"
)

In [ ]:
graph.query(
    "MATCH (n:__Entity__)"
    "RETURN count(*)"
)

In [ ]:
from graphdatascience import GraphDataScience

In [ ]:
# project graph
gds = GraphDataScience(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD), database=NEO4J_DATABASE

)

In [ ]:
gds.graph.drop('communities')

In [ ]:
G, result = gds.graph.project(
    "communities",  #  Graph name
    "__Entity__",  #  Node projection
    {
        "_ALL_": {
            "type": "*",
            "orientation": "UNDIRECTED",
            "properties": {"weight": {"property": "*", "aggregation": "COUNT"}},
        }
    },
)

In [ ]:
wcc = gds.wcc.stats(G)
print(f"Component count: {wcc['componentCount']}")
print(f"Component distribution: {wcc['componentDistribution']}")

In [ ]:
wcc = gds.wcc.stats(G)
print(f"Component count: {wcc['componentCount']}")
print(f"Component distribution: {wcc['componentDistribution']}")

In [ ]:
result = gds.leiden.stream(
    G,
    includeIntermediateCommunities=True,
    relationshipWeightProperty="weight",
    randomSeed=42,
    concurrency=1,
    theta=0
)

In [ ]:
result

In [ ]:
import pandas as pd

pd.read_csv("../tmp/quick_analysis_may_26.csv")

In [ ]:
pd.read_csv("../tmp/quick_analysis_june3.csv")

In [ ]:
result

In [ ]:
# The 'communities' column contains lists, e.g., [15, 3, 1]
# We want to extract the last element (the final community ID) from each list.
final_community_ids = result['intermediateCommunityIds'].apply(lambda community_list: community_list[-1])

# Now, we can count the number of unique final community IDs
num_unique_communities = final_community_ids.nunique()

print(f"The number of unique communities is: {num_unique_communities}")